# Optimización de Modelos Conjunto Soleado por GMM

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
#Importaciones
import warnings
warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import MinMaxScaler

#Lectura de datos
datos = pd.read_excel('/content/drive/MyDrive/Tesina/03_Clusterizacion_CTNET.xlsx')
datos.head(24)

,Fecha,Generación,Temperatura,Humedad Relativa,Índice UV,Hora,Cluster KMeans,Cluster GMM
0,2022-09-01 00:00:00,0.000000,19,77,0,0,Noche,Noche
1,2022-09-01 01:00:00,0.000000,19,82,0,1,Noche,Noche
2,2022-09-01 02:00:00,0.000000,18,85,0,2,Noche,Noche
3,2022-09-01 03:00:00,0.000000,18,87,0,3,Noche,Noche
4,2022-09-01 04:00:00,0.000000,18,88,0,4,Noche,Noche
5,2022-09-01 05:00:00,0.000000,17,86,0,5,Noche,Noche
6,2022-09-01 06:00:00,0.000000,18,89,0,6,Soleado,Lluvioso
7,2022-09-01 07:00:00,6.584959,18,95,0,7,Soleado,Lluvioso
8,2022-09-01 08:00:00,560.422022,18,100,0,8,Soleado,Lluvioso
9,2022-09-01 09:00:00,7720.582326,18,100,1,9,Soleado,Lluvioso


In [3]:
datos["Generacion_prev_hour"] = datos["Generación"].shift(1)
datos["Generacion_prev_day"] = datos["Generación"].shift(24)
datos = datos.dropna(how="any", axis= 0)

Definimos X y y

In [4]:
datos_dia = datos[datos["Cluster KMeans"] == "Nublado"].copy()
datos_dia.head(10)

,Fecha,Generación,Temperatura,Humedad Relativa,Índice UV,Hora,Cluster KMeans,Cluster GMM,Generacion_prev_hour,Generacion_prev_day
36,2022-09-02 12:00:00,27523.885172,21,57,4,12,Nublado,Nublado,17036.043251,29196.986647
37,2022-09-02 13:00:00,20596.278869,23,45,5,13,Nublado,Nublado,27523.885172,25478.471342
38,2022-09-02 14:00:00,28500.000000,24,37,6,14,Nublado,Nublado,20596.278869,29057.585772
39,2022-09-02 15:00:00,24647.568577,26,33,5,15,Nublado,Nublado,28500.000000,30000.000000
40,2022-09-02 16:00:00,25500.000000,27,34,4,16,Nublado,Nublado,24647.568577,28062.328964
41,2022-09-02 17:00:00,24281.956494,28,36,2,17,Nublado,Nublado,25500.000000,28786.629243
42,2022-09-02 18:00:00,22733.515002,26,39,1,18,Nublado,Nublado,24281.956494,29900.303971
43,2022-09-02 19:00:00,11972.590689,25,44,1,19,Nublado,Nublado,22733.515002,18282.505369
60,2022-09-03 12:00:00,17723.695569,21,58,4,12,Nublado,Soleado,11246.659307,27523.885172
61,2022-09-03 13:00:00,20400.000000,23,48,5,13,Nublado,Nublado,17723.695569,20596.278869


In [5]:
columns = datos_dia.drop(columns=["Fecha", "Generación", "Cluster KMeans", "Cluster GMM"]).columns

In [6]:
X = datos_dia[columns]
X

,Temperatura,Humedad Relativa,Índice UV,Hora,Generacion_prev_hour,Generacion_prev_day
36,21,57,4,12,17036.043251,29196.986647
37,23,45,5,13,27523.885172,25478.471342
38,24,37,6,14,20596.278869,29057.585772
39,26,33,5,15,28500.000000,30000.000000
40,27,34,4,16,24647.568577,28062.328964
...,...,...,...,...,...,...
18281,26,31,4,16,25562.000000,25385.000000
18282,26,32,2,17,25386.000000,22664.000000
18283,25,33,1,18,22872.000000,15736.000000
18284,23,38,0,19,15825.000000,1407.000000


In [7]:
y = datos_dia[['Generación']]
y

,Generación
36,27523.885172
37,20596.278869
38,28500.000000
39,24647.568577
40,25500.000000
...,...
18281,25386.000000
18282,22872.000000
18283,15825.000000
18284,1450.000000


Dividimos entrenamiento, validación y prueba

In [8]:
train_size = int(0.7 * len(X))
val_size = int(0.85 * len(X))

In [9]:
# Entrenamiento, validación y prueba, 75, 15 y 15
X_train, y_train =  X.iloc[:train_size, :], y.iloc[:train_size, :]
X_val, y_val = X.iloc[train_size:val_size, :], y.iloc[train_size:val_size, :]
X_test, y_test = X.iloc[val_size:, :],  y.iloc[val_size:,:]

print(f'X_train: {len(X_train)}, y_train: {len(y_train)}')
print(f'X_val: {len(X_val)}, y_val: {len(y_val)}')
print(f'X_test: {len(X_test)}, y_test: {len(y_test)}')

X_train: 4641, y_train: 4641
X_val: 994, y_val: 994
X_test: 995, y_test: 995


## Escalar con MinMaxScaler

In [10]:
from sklearn.preprocessing import MinMaxScaler

In [11]:
x_scaler = MinMaxScaler().fit(X_train)
x_scaler

MinMaxScaler()

In [12]:
X_train_scaled = x_scaler.transform(X_train)
print(X_train_scaled)
print(X_train_scaled.shape)

[[0.43333333 0.7761194  0.28571429 0.4        0.56786811 0.97323289]
 [0.5        0.59701493 0.35714286 0.46666667 0.91746284 0.84928238]
 [0.53333333 0.47761194 0.42857143 0.53333333 0.68654263 0.96858619]
 ...
 [0.53333333 0.08955224 0.28571429 0.4        0.89256667 0.71003333]
 [0.6        0.04477612 0.28571429 0.46666667 0.89183333 0.6942    ]
 [0.66666667 0.02985075 0.28571429 0.53333333 0.884      0.67946667]]
(4641, 6)


In [13]:
X_train_scaled_df = pd.DataFrame(X_train_scaled, index=X_train.index, columns=X_train.columns)
X_train_scaled_df

,Temperatura,Humedad Relativa,Índice UV,Hora,Generacion_prev_hour,Generacion_prev_day
36,0.433333,0.776119,0.285714,0.400000,0.567868,0.973233
37,0.500000,0.597015,0.357143,0.466667,0.917463,0.849282
38,0.533333,0.477612,0.428571,0.533333,0.686543,0.968586
39,0.600000,0.417910,0.357143,0.600000,0.950000,1.000000
40,0.633333,0.432836,0.285714,0.666667,0.821586,0.935411
...,...,...,...,...,...,...
13283,0.266667,0.253731,0.142857,0.266667,0.261667,0.703733
13284,0.400000,0.149254,0.214286,0.333333,0.847233,0.716300
13285,0.533333,0.089552,0.285714,0.400000,0.892567,0.710033
13286,0.600000,0.044776,0.285714,0.466667,0.891833,0.694200


In [14]:
X_val_scaled = x_scaler.transform(X_val)
print(X_val_scaled)
print(X_val_scaled.shape)

[[0.7        0.02985075 0.21428571 0.6        0.88153333 0.68466667]
 [0.76666667 0.02985075 0.14285714 0.66666667 0.89283333 0.6928    ]
 [0.8        0.01492537 0.07142857 0.73333333 0.89383333 0.66686667]
 ...
 [0.73333333 0.41791045 0.85714286 0.4        0.94243333 0.95253333]
 [0.8        0.29850746 1.         0.46666667 0.94176667 0.95486667]
 [0.83333333 0.2238806  0.85714286 0.53333333 0.93916667 0.97816667]]
(994, 6)


In [15]:
X_val_scaled_df = pd.DataFrame(X_val_scaled, index=X_val.index, columns=X_val.columns)
X_val_scaled_df

,Temperatura,Humedad Relativa,Índice UV,Hora,Generacion_prev_hour,Generacion_prev_day
13288,0.700000,0.029851,0.214286,0.600000,0.881533,0.684667
13289,0.766667,0.029851,0.142857,0.666667,0.892833,0.692800
13290,0.800000,0.014925,0.071429,0.733333,0.893833,0.666867
13291,0.766667,0.014925,0.000000,0.800000,0.853433,0.491367
13292,0.700000,0.029851,0.000000,0.866667,0.514800,0.095967
...,...,...,...,...,...,...
15323,0.533333,0.805970,0.357143,0.266667,0.846533,0.940067
15324,0.633333,0.582090,0.642857,0.333333,0.911033,0.968733
15325,0.733333,0.417910,0.857143,0.400000,0.942433,0.952533
15326,0.800000,0.298507,1.000000,0.466667,0.941767,0.954867


In [16]:
X_test_scaled = x_scaler.transform(X_test)
print(X_test_scaled)
print(X_test_scaled.shape)

[[0.9        0.1641791  0.64285714 0.6        0.93993333 0.95946667]
 [0.96666667 0.11940299 0.35714286 0.66666667 0.9288     0.94446667]
 [0.93333333 0.11940299 0.21428571 0.73333333 0.8208     0.89293333]
 ...
 [0.56666667 0.41791045 0.07142857 0.8        0.7624     0.52453333]
 [0.5        0.49253731 0.         0.86666667 0.5275     0.0469    ]
 [0.46666667 0.59701493 0.         0.93333333 0.04833333 0.        ]]
(995, 6)


In [17]:
X_test_scaled_df = pd.DataFrame(X_test_scaled, index=X_test.index, columns=X_test.columns)
X_test_scaled_df

,Temperatura,Humedad Relativa,Índice UV,Hora,Generacion_prev_hour,Generacion_prev_day
15328,0.900000,0.164179,0.642857,0.600000,0.939933,0.959467
15329,0.966667,0.119403,0.357143,0.666667,0.928800,0.944467
15330,0.933333,0.119403,0.214286,0.733333,0.820800,0.892933
15331,0.900000,0.134328,0.142857,0.800000,0.788567,0.768000
15332,0.833333,0.179104,0.071429,0.866667,0.689433,0.349533
...,...,...,...,...,...,...
18281,0.600000,0.388060,0.285714,0.666667,0.852067,0.846167
18282,0.600000,0.402985,0.142857,0.733333,0.846200,0.755467
18283,0.566667,0.417910,0.071429,0.800000,0.762400,0.524533
18284,0.500000,0.492537,0.000000,0.866667,0.527500,0.046900


In [18]:
x_scaller_all = MinMaxScaler().fit(X)
print(x_scaller_all)

MinMaxScaler()


In [19]:
X_scaled = x_scaller_all.transform(X)
print(X_scaled)
print(X_scaled.shape)

[[0.41935484 0.76056338 0.28571429 0.4        0.56786811 0.97323289]
 [0.48387097 0.5915493  0.35714286 0.46666667 0.91746284 0.84928238]
 [0.51612903 0.47887324 0.42857143 0.53333333 0.68654263 0.96858619]
 ...
 [0.5483871  0.42253521 0.07142857 0.8        0.7624     0.52453333]
 [0.48387097 0.49295775 0.         0.86666667 0.5275     0.0469    ]
 [0.4516129  0.5915493  0.         0.93333333 0.04833333 0.        ]]
(6630, 6)


In [20]:
X_scaled_df = pd.DataFrame(X_scaled, index=X.index, columns=X.columns)
X_scaled_df

,Temperatura,Humedad Relativa,Índice UV,Hora,Generacion_prev_hour,Generacion_prev_day
36,0.419355,0.760563,0.285714,0.400000,0.567868,0.973233
37,0.483871,0.591549,0.357143,0.466667,0.917463,0.849282
38,0.516129,0.478873,0.428571,0.533333,0.686543,0.968586
39,0.580645,0.422535,0.357143,0.600000,0.950000,1.000000
40,0.612903,0.436620,0.285714,0.666667,0.821586,0.935411
...,...,...,...,...,...,...
18281,0.580645,0.394366,0.285714,0.666667,0.852067,0.846167
18282,0.580645,0.408451,0.142857,0.733333,0.846200,0.755467
18283,0.548387,0.422535,0.071429,0.800000,0.762400,0.524533
18284,0.483871,0.492958,0.000000,0.866667,0.527500,0.046900


In [21]:
y_scaler = MinMaxScaler().fit(y_train)
print(y_scaler)

MinMaxScaler()


In [22]:
y_train_scaled = y_scaler.transform(y_train)
print(y_train_scaled)
print(y_train_scaled.shape)

[[0.91746284]
 [0.68654263]
 [0.95      ]
 ...
 [0.89183333]
 [0.884     ]
 [0.88153333]]
(4641, 1)


In [23]:
y_train_scaled_df = pd.DataFrame(y_train_scaled, index=y_train.index, columns=y_train.columns)
y_train_scaled_df

,Generación
36,0.917463
37,0.686543
38,0.950000
39,0.821586
40,0.850000
...,...
13283,0.847233
13284,0.892567
13285,0.891833
13286,0.884000


In [24]:
y_val_scaled = y_scaler.transform(y_val)
print(y_val_scaled)
print(y_val_scaled.shape)

[[8.92833333e-01]
 [8.93833333e-01]
 [8.53433333e-01]
 [5.14800000e-01]
 [5.18333333e-02]
 [0.00000000e+00]
 [0.00000000e+00]
 [7.03733333e-01]
 [6.26766667e-01]
 [7.09333333e-01]
 [7.83833333e-01]
 [6.76500000e-01]
 [6.85366667e-01]
 [6.87533333e-01]
 [6.60733333e-01]
 [4.83800000e-01]
 [9.24333333e-02]
 [0.00000000e+00]
 [6.76500000e-01]
 [6.89666667e-01]
 [7.04466667e-01]
 [6.53366667e-01]
 [4.85666667e-01]
 [6.37866667e-01]
 [6.10033333e-01]
 [5.91933333e-01]
 [5.99333333e-01]
 [6.08266667e-01]
 [6.53366667e-01]
 [4.23333333e-01]
 [9.31666667e-02]
 [8.27966667e-01]
 [7.89800000e-01]
 [8.75533333e-01]
 [8.66100000e-01]
 [8.54000000e-01]
 [7.82533333e-01]
 [7.58700000e-01]
 [5.93666667e-01]
 [1.34566667e-01]
 [0.00000000e+00]
 [0.00000000e+00]
 [8.95366667e-01]
 [8.78733333e-01]
 [8.57166667e-01]
 [8.45600000e-01]
 [8.68233333e-01]
 [8.61400000e-01]
 [8.16733333e-01]
 [6.21266667e-01]
 [1.01333333e-01]
 [0.00000000e+00]
 [0.00000000e+00]
 [9.48333333e-01]
 [9.95633333e-01]
 [9.944666

In [25]:
y_val_scaled_df = pd.DataFrame(y_val_scaled, index=y_val.index, columns=y_val.columns)
y_val_scaled_df

,Generación
13288,0.892833
13289,0.893833
13290,0.853433
13291,0.514800
13292,0.051833
...,...
15323,0.911033
15324,0.942433
15325,0.941767
15326,0.939167


In [26]:
y_test_scaled = y_scaler.transform(y_test)
print(y_test_scaled)
print(y_test_scaled.shape)

[[0.9288    ]
 [0.8208    ]
 [0.78856667]
 [0.68943333]
 [0.35676667]
 [0.0301    ]
 [0.        ]
 [0.7288    ]
 [0.77463333]
 [0.7569    ]
 [0.75133333]
 [0.8674    ]
 [0.77346667]
 [0.85126667]
 [0.78006667]
 [0.69566667]
 [0.35676667]
 [0.0335    ]
 [0.        ]
 [0.8673    ]
 [0.94856667]
 [0.97976667]
 [0.9742    ]
 [0.98153333]
 [0.94706667]
 [0.94543333]
 [0.9458    ]
 [0.7962    ]
 [0.73963333]
 [0.37316667]
 [0.02813333]
 [0.        ]
 [0.84413333]
 [0.91103333]
 [0.94243333]
 [0.94176667]
 [0.94386667]
 [0.9725    ]
 [0.9308    ]
 [0.7296    ]
 [0.80933333]
 [0.78803333]
 [0.40506667]
 [0.03346667]
 [0.        ]
 [0.8243    ]
 [0.91103333]
 [0.943     ]
 [0.94176667]
 [0.94886667]
 [0.95573333]
 [0.95593333]
 [0.9241    ]
 [0.86676667]
 [0.76606667]
 [0.42396667]
 [0.0338    ]
 [0.        ]
 [0.8237    ]
 [0.91103333]
 [0.9482    ]
 [0.94176667]
 [0.9482    ]
 [0.96123333]
 [0.95666667]
 [0.91926667]
 [0.86676667]
 [0.6952    ]
 [0.37006667]
 [0.03346667]
 [0.        ]
 [0.91

In [27]:
y_test_scaled_df = pd.DataFrame(y_test_scaled, index=y_test.index, columns=y_test.columns)
y_test_scaled_df

,Generación
15328,0.928800
15329,0.820800
15330,0.788567
15331,0.689433
15332,0.356767
...,...
18281,0.846200
18282,0.762400
18283,0.527500
18284,0.048333


In [28]:
y_scaller_all = MinMaxScaler().fit(y)
print(y_scaller_all)

MinMaxScaler()


In [29]:
y_scaled = y_scaller_all.transform(y)
print(y_scaled)
print(y_scaled.shape)

[[0.91746284]
 [0.68654263]
 [0.95      ]
 ...
 [0.5275    ]
 [0.04833333]
 [0.        ]]
(6630, 1)


In [30]:
y_scaled_df = pd.DataFrame(y_scaled, index=y.index, columns=y.columns)
y_scaled_df

,Generación
36,0.917463
37,0.686543
38,0.950000
39,0.821586
40,0.850000
...,...
18281,0.846200
18282,0.762400
18283,0.527500
18284,0.048333


## Preparación para Redes Neuronales

In [31]:
import numpy as np
import pandas as pd

def create_sliding_window_with_index(data_X, data_y, lookback):
    X, y, indices = [], [], []

    # Asegurar que `data_y` tiene los mismos índices que `data_X`
    data_y = data_y.reindex(data_X.index)

    max_index = len(data_X) - lookback

    for i in range(max_index):
        X.append(data_X.iloc[i:i + lookback].values)  # Ventana de entrada

        # Obtener el índice correcto en `data_y`
        y_index = data_X.index[i + lookback]

        # Extraer el valor correspondiente de `data_y`
        if y_index in data_y.index:
            y_value = data_y.loc[y_index]
            if isinstance(y_value, pd.Series):  # Si devuelve una serie, extraer el valor
                y_value = y_value.iloc[0]
        else:
            y_value = np.nan  # Si no está, asignamos NaN

        y.append(y_value)
        indices.append(y_index)  # 🔹 Guardamos el índice original de `data_y`

    # Convertimos `X` en un array y `y` en DataFrame conservando sus índices originales
    X_array = np.array(X)
    y_df = pd.DataFrame(y, index=indices, columns=['y'])  # 🔹 Conservamos los índices originales

    return X_array, y_df


In [32]:
lookback = 48  # Puedes ajustar a 24, 72, etc.

# Aplicar la ventana deslizante a cada conjunto
X_train_windowed, y_train_windowed = create_sliding_window_with_index(X_train_scaled_df, y_train_scaled_df, lookback)
X_val_windowed, y_val_windowed = create_sliding_window_with_index(X_val_scaled_df, y_val_scaled_df, lookback)
X_test_windowed, y_test_windowed = create_sliding_window_with_index(X_test_scaled_df, y_test_scaled_df, lookback)


In [33]:
print(f'X_train: {X_train_windowed.shape}, y_train: {y_train_windowed.shape}')
print(f'X_val: {X_val_windowed.shape}, y_val: {y_val_windowed.shape}')
print(f'X_test: {X_test_windowed.shape}, y_test: {y_test_windowed.shape}')

X_train: (4593, 48, 6), y_train: (4593, 1)
X_val: (946, 48, 6), y_val: (946, 1)
X_test: (947, 48, 6), y_test: (947, 1)


## Optuna

In [34]:
pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.6/383.6 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.8/231.8 kB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.5/78.5 kB 7.5 MB/s eta 0:00:00


In [35]:
from lightgbm import LGBMRegressor
from sklearn.ensemble import RandomForestRegressor
import optuna
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import cross_val_score
import seaborn as sns
from sklearn.metrics import mean_absolute_percentage_error as mean_absolute_percentage_error
from sklearn.metrics import mean_absolute_error as mean_absolute_error
from sklearn.metrics import mean_squared_error as mean_squared_error
from sklearn.metrics import r2_score as r2_score
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping

### Optimización de LightGBM

In [36]:
import optuna
import numpy as np
import pandas as pd
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_squared_error

# Definir la función de optimización
def objective(trial):
    # Espacio de búsqueda para los hiperparámetros
    num_leaves = trial.suggest_int('num_leaves', 10, 1000)
    subsample = trial.suggest_float('subsample', 0.1, 1.0)
    colsample_bytree = trial.suggest_float('colsample_bytree', 0.1, 1.0)
    min_data_in_leaf = trial.suggest_int('min_data_in_leaf', 10, 100)

    # Modelo LightGBM con los hiperparámetros sugeridos
    model = LGBMRegressor(
        num_leaves=num_leaves,
        subsample=subsample,
        colsample_bytree=colsample_bytree,
        min_data_in_leaf=min_data_in_leaf,
        random_state=42
    )

    # Entrenamos el modelo con los datos de entrenamiento
    model.fit(X_train_scaled_df, y_train_scaled_df)

    # Predicciones en el conjunto de validación
    y_pred = model.predict(X_val_scaled_df)
    rmse = mean_squared_error(y_val_scaled_df, y_pred)

    return rmse  # Optuna minimizará este valor

# Ejecutar la optimización con 50 iteraciones
LightGBM_study = optuna.create_study(direction="minimize")
LightGBM_study.optimize(objective, n_trials=50)

# Obtener los mejores hiperparámetros
best_params = LightGBM_study.best_params
print("Mejores hiperparámetros:", best_params)

[I 2025-03-14 03:13:44,918] A new study created in memory with name: no-name-9a3ff8fa-2dda-459b-aeb8-a524888bd8e4
[I 2025-03-14 03:13:45,132] Trial 0 finished with value: 0.003833215926179182 and parameters: {'num_leaves': 86, 'subsample': 0.34849292732635223, 'colsample_bytree': 0.6577823852250162, 'min_data_in_leaf': 24}. Best is trial 0 with value: 0.003833215926179182.


[LightGBM] [Warning] min_data_in_leaf is set=24, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=24
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Warning] min_data_in_leaf is set=24, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=24
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000981 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 635
[LightGBM] [Info] Number of data points in the train set: 4641, number of used features: 6
[LightGBM] [Info] Start training from score 0.608742
[LightGBM] [Warning] min_data_in_leaf is set=24, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=24
[LightGBM] [Warning] min_data_in_leaf is set=96, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=96


[I 2025-03-14 03:13:45,224] Trial 1 finished with value: 0.0036339970483070688 and parameters: {'num_leaves': 893, 'subsample': 0.155728742319433, 'colsample_bytree': 0.5830236741420652, 'min_data_in_leaf': 96}. Best is trial 1 with value: 0.0036339970483070688.


[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Warning] min_data_in_leaf is set=96, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=96
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000131 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 635
[LightGBM] [Info] Number of data points in the train set: 4641, number of used features: 6
[LightGBM] [Info] Start training from score 0.608742
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM

[I 2025-03-14 03:13:45,373] Trial 2 finished with value: 0.0035460621284409406 and parameters: {'num_leaves': 999, 'subsample': 0.11644877197214226, 'colsample_bytree': 0.6542716362752501, 'min_data_in_leaf': 46}. Best is trial 2 with value: 0.0035460621284409406.


...

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

[I 2025-03-14 03:13:52,410] Trial 48 finished with value: 0.0038055565142243136 and parameters: {'num_leaves': 634, 'subsample': 0.1048014988276329, 'colsample_bytree': 0.9591306652892188, 'min_data_in_leaf': 37}. Best is trial 21 with value: 0.0033351437437482254.


[LightGBM] [Warning] min_data_in_leaf is set=37, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=37
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Warning] min_data_in_leaf is set=37, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=37
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000197 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 635
[LightGBM] [Info] Number of data points in the train set: 4641, number of used features: 6
[LightGBM] [Info] Start training from score 0.608742
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further split

[I 2025-03-14 03:13:52,473] Trial 49 finished with value: 0.003409521775775256 and parameters: {'num_leaves': 16, 'subsample': 0.607113548722454, 'colsample_bytree': 0.8536308746582677, 'min_data_in_leaf': 33}. Best is trial 21 with value: 0.0033351437437482254.


[LightGBM] [Warning] min_data_in_leaf is set=33, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=33
Mejores hiperparámetros: {'num_leaves': 10, 'subsample': 0.6705774866860973, 'colsample_bytree': 0.9907112668862691, 'min_data_in_leaf': 37}


### Random Forest

In [37]:
def objective(trial):
    # Definir los hiperparámetros a optimizar
    n_estimators = trial.suggest_int("n_estimators", 100, 500, step=50)
    max_depth = trial.suggest_int("max_depth", 10, 50, step=5)
    min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
    min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 10)
    bootstrap = trial.suggest_categorical("bootstrap", [True, False])

    # Definir el modelo con los hiperparámetros actuales
    RF_model = RandomForestRegressor(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        bootstrap=bootstrap,
        criterion="squared_error",
        random_state=0
    )

    # Entrenar el modelo
    RF_model.fit(X_train_scaled_df, y_train_scaled_df)

    # Predicciones en el conjunto de validación
    y_pred = RF_model.predict(X_val_scaled_df)
    rmse = mean_squared_error(y_val_scaled_df, y_pred)

    return rmse  # Optuna minimizará este valor

# Crear el estudio de optimización
RF_study = optuna.create_study(direction="minimize")
RF_study.optimize(objective, n_trials=50)  # Ejecutar 50 pruebas

# Imprimir los mejores hiperparámetros encontrados
print("Mejores hiperparámetros:", RF_study.best_params)

[I 2025-03-14 03:13:52,480] A new study created in memory with name: no-name-b9d9d36f-b179-45d0-b144-31f5afc79200
[I 2025-03-14 03:13:58,552] Trial 0 finished with value: 0.006383512683529774 and parameters: {'n_estimators': 300, 'max_depth': 15, 'min_samples_split': 8, 'min_samples_leaf': 2, 'bootstrap': False}. Best is trial 0 with value: 0.006383512683529774.
[I 2025-03-14 03:14:03,999] Trial 1 finished with value: 0.0037274273685884343 and parameters: {'n_estimators': 450, 'max_depth': 20, 'min_samples_split': 14, 'min_samples_leaf': 7, 'bootstrap': True}. Best is trial 1 with value: 0.0037274273685884343.
[I 2025-03-14 03:14:07,345] Trial 2 finished with value: 0.0036711195702908133 and parameters: {'n_estimators': 300, 'max_depth': 25, 'min_samples_split': 9, 'min_samples_leaf': 10, 'bootstrap': True}. Best is trial 2 with value: 0.0036711195702908133.
[I 2025-03-14 03:14:09,287] Trial 3 finished with value: 0.00631375868143921 and parameters: {'n_estimators': 100, 'max_depth': 2

Mejores hiperparámetros: {'n_estimators': 500, 'max_depth': 40, 'min_samples_split': 11, 'min_samples_leaf': 10, 'bootstrap': True}


### CTNET

In [38]:
def compile_and_fit(model, xtrain=X_train_windowed, ytrain=y_train_windowed, learning_rate=0.0001):
    model.compile(loss=[tf.keras.losses.MeanSquaredError()],
                  optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
                  metrics=[tf.keras.metrics.RootMeanSquaredError(), tf.keras.metrics.MeanAbsolutePercentageError(), tf.keras.metrics.MeanAbsoluteError()])

    history = model.fit(xtrain, ytrain, epochs=50,
                        batch_size=512, validation_split=0.2, verbose=1)
    return history

def Loss(train_loss, valid_loss):
    plt.plot(train_loss)
    plt.plot(valid_loss)
    plt.rcParams["figure.figsize"] = (15, 3)
    plt.title('Model Losses')
    plt.ylabel('Loss')
    plt.xlabel('Epoch')
    plt.legend(['Train Loss', 'Validation Loss'], loc='upper left')
    plt.savefig('out/loss_plot.png')
    plt.show()

def transformer_encoder(inputs, head_size, num_heads, ff_dim, dropout=0):
    x = layers.LayerNormalization()(inputs)
    x = layers.Conv1D(filters=ff_dim, kernel_size=1, activation="relu", padding="same")(x)
    x = layers.Conv1D(filters=128, kernel_size=2, activation="relu", padding="same")(x)
    x = layers.Conv1D(filters=inputs.shape[-1], kernel_size=1)(x)
    res = x + inputs
    norm_x = layers.LayerNormalization()(res)
    x = layers.MultiHeadAttention(key_dim=head_size, num_heads=num_heads, dropout=dropout)(norm_x, norm_x)
    res = x + inputs
    norm_x = layers.LayerNormalization()(res)
    return norm_x

# Construir modelo con hiperparámetros sugeridos
def build_model(input_shape, head_size, num_heads, ff_dim, num_transformer_blocks, mlp_units, dropout, mlp_dropout):
    inputs = tf.keras.Input(shape=input_shape)
    x = inputs

    for _ in range(num_transformer_blocks):
        x = transformer_encoder(x, head_size, num_heads, ff_dim, dropout)

    x = layers.MultiHeadAttention(key_dim=head_size, num_heads=num_heads, dropout=dropout)(x, x)
    x = layers.LayerNormalization(epsilon=1e-6)(x)
    x = layers.GlobalAveragePooling1D(data_format="channels_first")(x)

    for units in mlp_units:
        x = layers.Dense(units, activation="relu")(x)

    x = layers.Dropout(mlp_dropout)(x)
    outputs = layers.Dense(1)(x)
    return tf.keras.Model(inputs, outputs)

In [39]:
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

In [40]:
# Definir la función objetivo para Optuna
def objective(trial):
    # Hiperparámetros a optimizar
    head_size = trial.suggest_int("head_size", 2, 8)
    num_heads = trial.suggest_int("num_heads", 2, 8)
    ff_dim = trial.suggest_int("ff_dim", 16, 128, step=16)
    num_transformer_blocks = trial.suggest_int("num_transformer_blocks", 1, 5)
    mlp_units = [trial.suggest_int("mlp_units_1", 64, 512, step=64),
                 trial.suggest_int("mlp_units_2", 32, 256, step=32)]
    dropout = trial.suggest_float("dropout", 0.1, 0.5)
    mlp_dropout = trial.suggest_float("mlp_dropout", 0.1, 0.5)
    learning_rate = trial.suggest_loguniform("learning_rate", 1e-5, 1e-2)
    batch_size = trial.suggest_categorical("batch_size", [128, 256, 512])

    # Construir el modelo
    model = build_model(
        input_shape=(X_train_windowed.shape[1], X_train_windowed.shape[2]),
        head_size=head_size,
        num_heads=num_heads,
        ff_dim=ff_dim,
        num_transformer_blocks=num_transformer_blocks,
        mlp_units=mlp_units,
        dropout=dropout,
        mlp_dropout=mlp_dropout
    )

    # Compilar y entrenar el modelo
    model.compile(
        loss=tf.keras.losses.MeanSquaredError(),
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        metrics=[tf.keras.metrics.RootMeanSquaredError()]
    )

    history = model.fit(
        X_train_windowed, y_train_windowed,
        epochs=100,  # Se puede detener antes con Early Stopping
        batch_size=batch_size,  # 🔹 Batch size optimizado
        validation_data=(X_val_windowed, y_val_windowed),  # 🔹 Validación en cada época
        callbacks=[early_stop],  # 🔹 Early Stopping activado
        verbose=0
    )

    # Obtener pérdida de validación
    val_loss = min(history.history['val_loss'])
    return val_loss  # Minimizar MSE

# Crear el estudio de optimización
CTNET_study = optuna.create_study(direction="minimize")
CTNET_study.optimize(objective, n_trials=50)  # Ejecutar 50 pruebas

# Imprimir los mejores hiperparámetros encontrados
print("Mejores hiperparámetros:", CTNET_study.best_params)


[I 2025-03-14 03:18:05,998] A new study created in memory with name: no-name-6fcad712-75a3-4d48-be1d-e92ed35272dd
[I 2025-03-14 03:19:02,888] Trial 0 finished with value: 0.04209723323583603 and parameters: {'head_size': 5, 'num_heads': 2, 'ff_dim': 32, 'num_transformer_blocks': 2, 'mlp_units_1': 320, 'mlp_units_2': 32, 'dropout': 0.2834417621934571, 'mlp_dropout': 0.3918526042440692, 'learning_rate': 0.0001145885594966619, 'batch_size': 128}. Best is trial 0 with value: 0.04209723323583603.
[I 2025-03-14 03:19:49,186] Trial 1 finished with value: 0.12246109545230865 and parameters: {'head_size': 2, 'num_heads': 7, 'ff_dim': 32, 'num_transformer_blocks': 1, 'mlp_units_1': 448, 'mlp_units_2': 128, 'dropout': 0.4823495503922436, 'mlp_dropout': 0.32802147766954015, 'learning_rate': 4.5625940218394675e-05, 'batch_size': 256}. Best is trial 0 with value: 0.04209723323583603.
[I 2025-03-14 03:21:14,669] Trial 2 finished with value: 0.017130950465798378 and parameters: {'head_size': 8, 'num_h

Mejores hiperparámetros: {'head_size': 6, 'num_heads': 2, 'ff_dim': 48, 'num_transformer_blocks': 5, 'mlp_units_1': 448, 'mlp_units_2': 256, 'dropout': 0.3360822470360483, 'mlp_dropout': 0.32432206414310666, 'learning_rate': 0.001237208136295941, 'batch_size': 128}


### Forescasting

In [41]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import InputLayer, Conv1D, BatchNormalization, MaxPooling1D, Bidirectional, LSTM, Dropout, Dense
from tensorflow.keras.optimizers import Adam

# Definir la función objetivo para Optuna
def objective(trial):
    # Hiperparámetros a optimizar
    filters = trial.suggest_categorical("filters", [32, 64, 128])
    kernel_size = trial.suggest_int("kernel_size", 2, 5)
    lstm_units_1 = trial.suggest_categorical("lstm_units_1", [64, 128, 256])
    lstm_units_2 = trial.suggest_categorical("lstm_units_2", [32, 64, 128])
    lstm_units_3 = trial.suggest_categorical("lstm_units_3", [16, 32, 64])
    dropout_lstm = trial.suggest_float("dropout_lstm", 0.1, 0.5)
    dropout_dense = trial.suggest_float("dropout_dense", 0.1, 0.5)
    learning_rate = trial.suggest_loguniform("learning_rate", 1e-4, 1e-2)
    batch_size = trial.suggest_categorical("batch_size", [128, 256, 512])  # 🔹 Optimización del batch size

    # Construcción del modelo
    model = Sequential()
    model.add(InputLayer((X_train_windowed.shape[1], X_train_windowed.shape[2])))

    # CNN
    model.add(Conv1D(filters=filters, kernel_size=kernel_size, padding='same', activation='relu'))
    model.add(BatchNormalization())
    model.add(MaxPooling1D(pool_size=2))

    # BiLSTM
    model.add(Bidirectional(LSTM(lstm_units_1, return_sequences=True)))
    model.add(Bidirectional(LSTM(lstm_units_2, return_sequences=True)))
    model.add(Dropout(dropout_lstm))
    model.add(Bidirectional(LSTM(lstm_units_3, return_sequences=False)))

    # Normalización y Dropout
    model.add(BatchNormalization())
    model.add(Dropout(dropout_dense))

    # Capas Densas
    model.add(Dense(16, activation='relu'))
    model.add(Dense(1, activation='relu'))

    # Compilar el modelo
    model.compile(
        loss=tf.keras.losses.MeanSquaredError(),
        optimizer=Adam(learning_rate=learning_rate),
        metrics=[tf.keras.metrics.RootMeanSquaredError()]
    )

    # Entrenar el modelo con validación
    history = model.fit(
        X_train_windowed, y_train_windowed,
        epochs=100,  # Se puede detener antes con Early Stopping
        batch_size=batch_size,  # 🔹 Batch size optimizado
        validation_data=(X_val_windowed, y_val_windowed),  # 🔹 Validación en cada época
        callbacks=[early_stop],  # 🔹 Early Stopping activado
        verbose=0
    )

    # Obtener la mejor pérdida de validación
    val_loss = min(history.history['val_loss'])
    return val_loss  # Optuna minimizará este valor

# Crear el estudio de optimización
Forecasting_study = optuna.create_study(direction="minimize")
Forecasting_study.optimize(objective, n_trials=50, n_jobs=-1)  # Ejecutar 50 pruebas

# Imprimir los mejores hiperparámetros encontrados
print("Mejores hiperparámetros:", Forecasting_study.best_params)

[I 2025-03-14 04:14:55,131] A new study created in memory with name: no-name-5634438c-9fbb-4581-8aaa-62deae5204df
[I 2025-03-14 04:16:24,746] Trial 11 finished with value: 0.156172975897789 and parameters: {'filters': 128, 'kernel_size': 5, 'lstm_units_1': 256, 'lstm_units_2': 64, 'lstm_units_3': 32, 'dropout_lstm': 0.4586887589315084, 'dropout_dense': 0.3722642180326162, 'learning_rate': 0.0007914457778434024, 'batch_size': 256}. Best is trial 11 with value: 0.156172975897789.
[I 2025-03-14 04:16:36,071] Trial 12 finished with value: 0.20963075757026672 and parameters: {'filters': 64, 'kernel_size': 2, 'lstm_units_1': 256, 'lstm_units_2': 128, 'lstm_units_3': 32, 'dropout_lstm': 0.3264048264900261, 'dropout_dense': 0.123827987649261, 'learning_rate': 0.001874078805325109, 'batch_size': 256}. Best is trial 11 with value: 0.156172975897789.
[I 2025-03-14 04:16:47,475] Trial 13 finished with value: 0.3738705813884735 and parameters: {'filters': 64, 'kernel_size': 5, 'lstm_units_1': 64, '

Mejores hiperparámetros: {'filters': 128, 'kernel_size': 3, 'lstm_units_1': 64, 'lstm_units_2': 64, 'lstm_units_3': 16, 'dropout_lstm': 0.1907120591332447, 'dropout_dense': 0.3841859555154085, 'learning_rate': 0.005337799002681695, 'batch_size': 128}


### Photovoltaic

In [42]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv1D, MaxPooling1D, Bidirectional, GRU, MultiHeadAttention
from tensorflow.keras.layers import Flatten, Dropout, Dense
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import mean_squared_error

# Definir la función objetivo para Optuna
def objective(trial):
    # Hiperparámetros a optimizar
    filters_1 = trial.suggest_categorical("filters_1", [32, 64, 128])
    filters_2 = trial.suggest_categorical("filters_2", [64, 128, 256])
    kernel_size = trial.suggest_int("kernel_size", 2, 5)
    gru_units = trial.suggest_categorical("gru_units", [32, 64, 128])
    num_heads = trial.suggest_categorical("num_heads", [2, 4, 8])
    dropout_rate = trial.suggest_float("dropout_rate", 0.2, 0.5)
    dense_units_1 = trial.suggest_categorical("dense_units_1", [32, 64, 128])
    dense_units_2 = trial.suggest_categorical("dense_units_2", [16, 32, 64])
    learning_rate = trial.suggest_loguniform("learning_rate", 1e-4, 1e-2)
    batch_size = trial.suggest_categorical("batch_size", [128, 256, 512])

    # Definir el modelo
    inputs = Input(shape=(X_train_windowed.shape[1], X_train_windowed.shape[2]))

    # Primera capa CNN
    x = Conv1D(filters=filters_1, kernel_size=kernel_size, padding='same', activation='relu')(inputs)
    x = MaxPooling1D(pool_size=2)(x)

    # Segunda capa CNN
    x = Conv1D(filters=filters_2, kernel_size=kernel_size, padding='same', activation='relu')(x)
    x = MaxPooling1D(pool_size=2)(x)

    # Capa BiGRU
    x = Bidirectional(GRU(gru_units, return_sequences=True))(x)

    # Atención MultiHead
    attention = MultiHeadAttention(num_heads=num_heads, key_dim=128)(x, x)

    # Aplanar y Dropout
    x = Flatten()(attention)
    x = Dropout(dropout_rate)(x)
    x = Dense(dense_units_1, activation="relu", kernel_regularizer=l2(0.01))(x)
    x = Dense(dense_units_2, activation="relu")(x)

    # Capa de salida
    outputs = Dense(1, activation="linear")(x)

    # Construcción del modelo
    model = Model(inputs=inputs, outputs=outputs)

    # Compilar el modelo
    model.compile(
        loss=tf.keras.losses.MeanSquaredError(),
        optimizer=Adam(learning_rate=learning_rate),
        metrics=[tf.keras.metrics.RootMeanSquaredError()]
    )

    # Early Stopping
    early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1)

    # Entrenar el modelo
    history = model.fit(
        X_train_windowed, y_train_windowed,
        epochs=100,  # Se detendrá con Early Stopping
        batch_size=batch_size,
        validation_data=(X_val_windowed, y_val_windowed),
        callbacks=[early_stop],
        verbose=0
    )

    # Obtener la mejor pérdida de validación
    val_loss = min(history.history['val_loss'])
    return val_loss  # Optuna minimizará este valor

# Crear el estudio de optimización
Photovoltaic_study = optuna.create_study(direction="minimize")
Photovoltaic_study.optimize(objective, n_trials=50, n_jobs=-1)  # Ejecutar 50 pruebas

# Imprimir los mejores hiperparámetros encontrados
print("Mejores hiperparámetros:", Photovoltaic_study.best_params)

[I 2025-03-14 04:26:46,467] A new study created in memory with name: no-name-9a82c53f-95fe-4f4e-b352-bf8aff70762b


Epoch 25: early stopping
Restoring model weights from the end of the best epoch: 15.


[I 2025-03-14 04:28:36,051] Trial 6 finished with value: 0.013736714608967304 and parameters: {'filters_1': 64, 'filters_2': 256, 'kernel_size': 2, 'gru_units': 32, 'num_heads': 2, 'dropout_rate': 0.3938452117846962, 'dense_units_1': 64, 'dense_units_2': 64, 'learning_rate': 0.003771794003679529, 'batch_size': 256}. Best is trial 6 with value: 0.013736714608967304.


Epoch 74: early stopping
Restoring model weights from the end of the best epoch: 64.


[I 2025-03-14 04:29:02,955] Trial 0 finished with value: 0.016442205756902695 and parameters: {'filters_1': 32, 'filters_2': 64, 'kernel_size': 5, 'gru_units': 128, 'num_heads': 4, 'dropout_rate': 0.4417461483219919, 'dense_units_1': 128, 'dense_units_2': 16, 'learning_rate': 0.001184161135227899, 'batch_size': 512}. Best is trial 6 with value: 0.013736714608967304.


...

Epoch 62: early stopping
Restoring model weights from the end of the best epoch: 52.


[I 2025-03-14 04:37:55,510] Trial 47 finished with value: 0.015617715194821358 and parameters: {'filters_1': 128, 'filters_2': 256, 'kernel_size': 3, 'gru_units': 32, 'num_heads': 8, 'dropout_rate': 0.46452907294029555, 'dense_units_1': 128, 'dense_units_2': 32, 'learning_rate': 0.0003328298928570522, 'batch_size': 128}. Best is trial 18 with value: 0.012901460751891136.


Epoch 67: early stopping
Restoring model weights from the end of the best epoch: 57.


[I 2025-03-14 04:37:57,383] Trial 46 finished with value: 0.014311643317341805 and parameters: {'filters_1': 128, 'filters_2': 256, 'kernel_size': 3, 'gru_units': 32, 'num_heads': 8, 'dropout_rate': 0.4755903637807232, 'dense_units_1': 128, 'dense_units_2': 32, 'learning_rate': 0.00036395514235322096, 'batch_size': 128}. Best is trial 18 with value: 0.012901460751891136.


Mejores hiperparámetros: {'filters_1': 128, 'filters_2': 64, 'kernel_size': 2, 'gru_units': 32, 'num_heads': 2, 'dropout_rate': 0.2547668056441918, 'dense_units_1': 128, 'dense_units_2': 64, 'learning_rate': 0.0004315906506319696, 'batch_size': 512}
